# FPL data preparation

Prepared 113,260 historical player-fixture records, 3,268 player-season identities, 80 team-season rows and 1,520 fixtures for 2022/23–2025/26. Historical `P_off` and `P_adj` retain their existing values under the [versioned target policy](../docs/fpl_rules.md#adjusted-target-used-in-notebook-1). All 42,705 historical outfield appearances have complete defensive counts.

The complete collected 2026/27 batch supplies 6,562 player-fixture records for 672 players across 100 finalized fixtures through GW 10. Names, positions and match-time clubs are joined using native FPL IDs. Current `P_off` remains exactly as collected, totalling 9,145 points; it is not recalculated. Snapshots remain separate and retain actual collection times.

Keys, fixture context, source checksums, nulls, duplicates and totals are checked. All 7,392 current reconciliation checks pass. Historical source differences and provider disagreements remain in audits. EDA tables are saved in `data/processed/fpl_tables.pkl`, with `fpl_audits.pkl` and provenance in `fpl_manifest.json`. Raw files remain unchanged.

## Data import

Collection commands, saved batches and source-table definitions are described in the
[collection guide](../docs/fpl_collection_learning.md).

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 80)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

# Resolve paths from the project root or notebooks directory
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_preparation import (
    MATCH_KEY, export_prepared_data, prepare_collected_batch, prepare_fpl_tables,
    prepare_player_matches, profile_columns,
    profile_gameweeks, profile_tables, read_fpl_sources, reconcile_player_totals,
    require_keys, stack_seasons, validate_fpl_relations,
)
from src.defensive_contributions import ADJUSTED_RULES
from src.fotmob_identity import (
    PLAYER_NAME_ALIASES, build_fixture_identity_map, build_player_identity_map,
    build_team_identity_map, external_match_inputs, load_fotmob_identity_lookups,
    player_matching_report,
)

ARCHIVE_DIR = PROJECT_ROOT / "data/epl-premier-league-dataset"
RAW_DIR = ARCHIVE_DIR / "raw_data/fpl_api"
SEASONS = ["2022_2023", "2023_2024", "2024_2025", "2025_2026"]
MERGED_SEASON = "2024_2025"
P_ADJ_RULES = ADJUSTED_RULES

## Source inventory

Load every CSV and both saved JSON responses from `raw_data/fpl_api/` separately. Record UTF-8/CP1252 encodings and expose JSON record lists without flattening nested values. Filenames describe source families, not verified collection methods.

In [2]:
tables, source_inventory, json_structure = read_fpl_sources(RAW_DIR)
bootstrap_players = tables["fpl_bootstrap_latest.json / elements"]
display(source_inventory)
display(source_inventory.groupby("format").size().rename("files").to_frame())

,filename,format,season,encoding
0,fpl_bootstrap_latest.json,json,snapshot; inspect metadata,utf-8
1,fpl_cleaned_players_2016_2017.csv,csv,2016_2017,cp1252
2,fpl_cleaned_players_2017_2018.csv,csv,2017_2018,utf-8
3,fpl_cleaned_players_2018_2019.csv,csv,2018_2019,utf-8
4,fpl_cleaned_players_2019_2020.csv,csv,2019_2020,utf-8
5,fpl_cleaned_players_2020_2021.csv,csv,2020_2021,utf-8
6,fpl_cleaned_players_2021_2022.csv,csv,2021_2022,utf-8
7,fpl_cleaned_players_2022_2023.csv,csv,2022_2023,utf-8
8,fpl_cleaned_players_2023_2024.csv,csv,2023_2024,utf-8
9,fpl_cleaned_players_2024_2025.csv,csv,2024_2025,utf-8


,files
format,
csv,33
json,2


## Table structure and join keys

| Source | Row level | Candidate key within a file |
|---|---|---|
| `fpl_cleaned_players_<season>.csv` | Player or manager season summary | No player ID supplied |
| `fpl_merged_gw_<season>.csv` | Player or manager in one fixture | `element, fixture` |
| `fpl_fixtures_<season>.csv` | Fixture | `id` |
| `fpl_teams_<season>.csv` | Team | `id` |
| Bootstrap `elements` | Player at the saved snapshot | `id` |
| Bootstrap `teams`, `events`, `element_types` | Team, gameweek or position | `id` |
| Fixtures JSON | Fixture at the saved snapshot | `id` |

Scope historical FPL keys by season. Names are descriptive attributes; ID-less season summaries remain separate from the identity-bearing match records.

## Rows, columns, and data types

Choose `TABLE_NAME` to inspect another source. CSV types are inferred; JSON numeric/date strings remain unparsed until preparation.

In [3]:
quality_summary, column_profile = profile_tables(tables)
table_summary = quality_summary[["table", "rows", "columns"]]
display(table_summary)

# Change this name to inspect another CSV or JSON record list
TABLE_NAME = "fpl_cleaned_players_2024_2025.csv"
table = tables[TABLE_NAME]
print(TABLE_NAME, "shape:", table.shape)
display(table.head())

,table,rows,columns
0,fpl_bootstrap_latest.json / chips,8,7
1,fpl_bootstrap_latest.json / events,38,29
2,fpl_bootstrap_latest.json / phases,11,5
3,fpl_bootstrap_latest.json / teams,20,22
4,fpl_bootstrap_latest.json / element_stats,26,2
5,fpl_bootstrap_latest.json / element_types,4,13
6,fpl_bootstrap_latest.json / elements,667,109
7,fpl_cleaned_players_2016_2017.csv,683,17
8,fpl_cleaned_players_2017_2018.csv,647,18
9,fpl_cleaned_players_2018_2019.csv,624,18


fpl_cleaned_players_2024_2025.csv shape: (804, 19)


,first_name,second_name,goals_scored,assists,total_points,minutes,goals_conceded,creativity,influence,threat,bonus,bps,ict_index,clean_sheets,red_cards,yellow_cards,selected_by_percent,now_cost,element_type
0,Fábio,Ferreira Vieira,0,0,0,0,0,0.0000,0.0000,0.0000,0,0,0.0000,0,0,0,0.0000,54,MID
1,Gabriel,Fernando de Jesus,3,2,42,600,5,119.5000,154.4000,255.0000,6,152,52.6000,2,0,4,1.0000,65,FWD
2,Gabriel,dos Santos Magalhães,3,2,117,2363,22,208.8000,584.6000,287.0000,9,459,108.2000,10,0,4,12.2000,61,DEF
3,Kai,Havertz,9,3,97,1872,20,269.0000,467.6000,711.0000,14,343,144.8000,7,0,5,6.2000,77,FWD
4,Karl,Hein,0,0,0,0,0,0.0000,0.0000,0.0000,0,0,0.0000,0,0,0,0.0000,40,GK


## Data quality checks

Report types, non-null counts, nulls and distinct values for every table. Count exact duplicates separately from repeated candidate keys; neither repeated names nor conflicting keys justify automatic deletion. Nested lists/dictionaries are serialized only in temporary comparison copies.

In [4]:
display(quality_summary)
selected_profile = column_profile.loc[column_profile["table"].eq(TABLE_NAME)].reset_index(drop=True)
with pd.option_context("display.max_rows", None):
    display(selected_profile)

# Examples of exact duplicate records and repeated names
matches = tables["fpl_merged_gw_2025_2026.csv"]
display(matches.loc[matches.duplicated(keep=False), ["name", "element", "fixture", "GW", "minutes", "total_points"]].head(6))
players = tables["fpl_cleaned_players_2022_2023.csv"]
display(players.loc[
    players.duplicated(["first_name", "second_name"], keep=False),
    ["first_name", "second_name", "element_type", "minutes", "total_points"],
])

,table,rows,columns,null_cells,extra_exact_duplicates,candidate_key,null_key_rows,extra_key_duplicates
0,fpl_bootstrap_latest.json / chips,8,7,0,0,id,0,0
1,fpl_bootstrap_latest.json / events,38,29,296,0,id,0,0
2,fpl_bootstrap_latest.json / phases,11,5,8,0,id,0,0
3,fpl_bootstrap_latest.json / teams,20,22,60,0,id,0,0
4,fpl_bootstrap_latest.json / element_stats,26,2,0,0,not supplied / not assigned,<NA>,<NA>
5,fpl_bootstrap_latest.json / element_types,4,13,8,0,id,0,0
6,fpl_bootstrap_latest.json / elements,667,109,4239,0,id,0,0
7,fpl_cleaned_players_2016_2017.csv,683,17,0,0,not supplied / not assigned,<NA>,<NA>
8,fpl_cleaned_players_2017_2018.csv,647,18,0,0,not supplied / not assigned,<NA>,<NA>
9,fpl_cleaned_players_2018_2019.csv,624,18,0,0,not supplied / not assigned,<NA>,<NA>


,table,column,dtype,non_null,nulls,null_share,unique_non_null
0,fpl_cleaned_players_2024_2025.csv,first_name,object,804,0,0.0000,540
1,fpl_cleaned_players_2024_2025.csv,second_name,object,804,0,0.0000,763
2,fpl_cleaned_players_2024_2025.csv,goals_scored,int64,804,0,0.0000,22
3,fpl_cleaned_players_2024_2025.csv,assists,int64,804,0,0.0000,15
4,fpl_cleaned_players_2024_2025.csv,total_points,int64,804,0,0.0000,161
5,fpl_cleaned_players_2024_2025.csv,minutes,int64,804,0,0.0000,483
6,fpl_cleaned_players_2024_2025.csv,goals_conceded,int64,804,0,0.0000,70
7,fpl_cleaned_players_2024_2025.csv,creativity,float64,804,0,0.0000,474
8,fpl_cleaned_players_2024_2025.csv,influence,float64,804,0,0.0000,479
9,fpl_cleaned_players_2024_2025.csv,threat,float64,804,0,0.0000,306


,name,element,fixture,GW,minutes,total_points
7,Ben Gannon-Doak,391,1,1,0,0
91,Junior Kroupi,100,1,1,1,1
291,Ben Gannon-Doak,391,1,1,0,0
638,Junior Kroupi,100,1,1,1,1
784,Junior Kroupi,100,12,2,2,1
1343,Junior Kroupi,100,12,2,2,1


,first_name,second_name,element_type,minutes,total_points
462,Ben,Davies,DEF,0,0
678,Ben,Davies,DEF,2284,99


## What does `fpl_merged_gw` mean?

`gw` means **gameweek**. Each file stacks a season's gameweek records vertically. One row represents an FPL player or manager in one fixture; `GW`/`round` label the gameweek. Double/triple gameweeks contain distinct fixtures, so `element, GW` is not a match key.

The filename does not establish an external-source merge. Local collector code is unavailable, and FPL itself supplies the expected-stat fields checked below. The 2019/20 repeated-key example is an inspection finding outside our preparation period.

In [5]:
gameweek_structure = profile_gameweeks(tables)
display(gameweek_structure)

matches = tables["fpl_merged_gw_2024_2025.csv"]
display(matches.loc[
    matches["element"].eq(328) & matches["GW"].eq(24),
    ["name", "element", "fixture", "GW", "round", "kickoff_time"],
])
older_matches = tables["fpl_merged_gw_2019_2020.csv"]
repeated = older_matches.loc[older_matches.duplicated(["element", "fixture"], keep=False)]
first_key = repeated[["element", "fixture"]].iloc[0]
display(repeated.loc[
    repeated["element"].eq(first_key["element"]) & repeated["fixture"].eq(first_key["fixture"]),
    ["name", "element", "fixture", "GW", "round", "kickoff_time", "team_h_score", "team_a_score"],
])

,filename,rows,columns,first_GW,last_GW,distinct_GWs,player_GW_groups_with_multiple_fixtures,max_distinct_fixtures_per_player_GW,extra_exact_duplicates,extra_player_fixture_keys
0,fpl_merged_gw_2016_2017.csv,23679,56,1,38,38,573,2,0,0
1,fpl_merged_gw_2017_2018.csv,22467,56,1,38,38,670,2,0,0
2,fpl_merged_gw_2018_2019.csv,21790,56,1,38,38,656,2,0,0
3,fpl_merged_gw_2019_2020.csv,22560,33,1,47,38,247,2,0,59
4,fpl_merged_gw_2020_2021.csv,24365,36,1,38,38,1437,3,0,0
5,fpl_merged_gw_2021_2022.csv,25447,36,1,38,38,2217,2,0,0
6,fpl_merged_gw_2022_2023.csv,26505,41,1,38,37,1548,2,0,0
7,fpl_merged_gw_2023_2024.csv,29725,41,1,38,38,983,2,0,0
8,fpl_merged_gw_2024_2025.csv,27605,49,1,38,38,374,2,0,0
9,fpl_merged_gw_2025_2026.csv,29757,46,1,38,38,409,2,10,10


,name,element,fixture,GW,round,kickoff_time
15830,Mohamed Salah,328,232,24,24,2025-02-01T15:00:00Z
15831,Mohamed Salah,328,144,24,24,2025-02-12T19:30:00Z


,name,element,fixture,GW,round,kickoff_time,team_h_score,team_a_score
15874,John_Stones_207,207,275,29,29,2020-03-11T19:30:00Z,NaN,NaN
16929,John_Stones_207,207,275,39,39,2020-06-17T19:15:00Z,3.0000,0.0000


### Columns, data types and meaning

| Group | Main fields |
|---|---|
| Identity | `element`, `name`, `position`, `team` |
| Fixture context | `fixture`, `opponent_team`, `was_home`, `kickoff_time`, `GW`, `round` |
| Football outcomes | `minutes`, `starts`, goals, assists, clean sheets, saves, cards and penalties |
| Expected statistics | `expected_goals`, `expected_assists`, `expected_goal_involvements`, `expected_goals_conceded` |
| Fantasy statistics | `total_points`, `bonus`, `bps`, `xP`, `value`, `selected`, transfers |
| FPL indices | `influence`, `creativity`, `threat`, `ict_index` |
| Other context | `team_h_score`, `team_a_score`, `modified` where supplied |
| Season-specific fields | `mng_*`, defensive actions and contributions |

Inspect the selected season's original rows and column profile below. `value = 105` means £10.5m; `selected` is a count. Expected statistics describe recorded chances, not the next fixture. Index definitions and `modified` semantics require source documentation.

In [6]:
merged_table = tables[f"fpl_merged_gw_{MERGED_SEASON}.csv"]
merged_column_profile = profile_columns(merged_table, detailed=True)
print("Selected season:", MERGED_SEASON, "shape:", merged_table.shape)
with pd.option_context("display.max_rows", None):
    display(merged_table.head(3).T)
    display(merged_column_profile)
display(merged_table["position"].value_counts(dropna=False).rename("rows").to_frame())
print("Rows where GW and round differ:", int(merged_table["GW"].ne(merged_table["round"]).sum()))

Selected season: 2024_2025 shape: (27605, 49)


,0,1,2
name,Alex Scott,Carlos Miguel dos Santos Pereira,Tomiyasu Takehiro
position,MID,GK,DEF
team,Bournemouth,Nott'm Forest,Arsenal
xP,1.6000,2.2000,0.0000
assists,0,0,0
bonus,0,0,0
bps,11,0,0
clean_sheets,0,0,0
creativity,12.8000,0.0000,0.0000
element,77,427,22


,column,dtype,non_null,nulls,null_share,unique_non_null,numeric_min,numeric_max,example_values
0,name,object,27605,0,0.0000,805,<NA>,<NA>,"'Alex Scott', 'Carlos Miguel dos Santos Pereir..."
1,position,object,27605,0,0.0000,5,<NA>,<NA>,"'MID', 'GK', 'DEF'"
2,team,object,27605,0,0.0000,20,<NA>,<NA>,"'Bournemouth', ""Nott'm Forest"", 'Arsenal'"
3,xP,float64,27605,0,0.0000,154,-2.2000,26.4000,"1.6, 2.2, 0.0"
4,assists,int64,27605,0,0.0000,4,0,3,"0, 1, 3"
5,bonus,int64,27605,0,0.0000,4,0,3,"0, 3, 1"
6,bps,int64,27605,0,0.0000,105,-25,97,"11, 0, -2"
7,clean_sheets,int64,27605,0,0.0000,2,0,1,"0, 1"
8,creativity,float64,27605,0,0.0000,653,0.0000,114.3000,"12.8, 0.0, 14.0"
9,element,int64,27605,0,0.0000,804,1,804,"77, 427, 22"


,rows
position,
MID,12250
DEF,9146
FWD,3018
GK,2869
AM,322


Rows where GW and round differ: 0


### Differences between the selected seasons

Compare column presence and inferred types before stacking observations. `not supplied` means a field is absent, not zero. Manager fields, `modified` and defensive fields account for the main schema differences.

In [7]:
merged_schema = pd.DataFrame({
    season: tables[f"fpl_merged_gw_{season}.csv"].dtypes.astype(str) for season in SEASONS
}).fillna("not supplied")
with pd.option_context("display.max_rows", None):
    display(merged_schema)
display(merged_schema.loc[merged_schema.nunique(axis=1).gt(1)])

,2022_2023,2023_2024,2024_2025,2025_2026
GW,int64,int64,int64,int64
assists,int64,int64,int64,int64
bonus,int64,int64,int64,int64
bps,int64,int64,int64,int64
clean_sheets,int64,int64,int64,int64
clearances_blocks_interceptions,not supplied,not supplied,not supplied,int64
creativity,float64,float64,float64,float64
defensive_contribution,not supplied,not supplied,not supplied,int64
element,int64,int64,int64,int64
expected_assists,float64,float64,float64,float64


,2022_2023,2023_2024,2024_2025,2025_2026
clearances_blocks_interceptions,not supplied,not supplied,not supplied,int64
defensive_contribution,not supplied,not supplied,not supplied,int64
mng_clean_sheets,not supplied,not supplied,float64,not supplied
mng_draw,not supplied,not supplied,float64,not supplied
mng_goals_scored,not supplied,not supplied,float64,not supplied
mng_loss,not supplied,not supplied,float64,not supplied
mng_underdog_draw,not supplied,not supplied,float64,not supplied
mng_underdog_win,not supplied,not supplied,float64,not supplied
mng_win,not supplied,not supplied,float64,not supplied
modified,not supplied,not supplied,bool,bool


### FPL fields and source provenance

Matching field names below show that these measures are available from FPL. They do not prove identical definitions, values or retrieval times across files.

In [8]:
fields = [
    "goals_scored", "assists", "minutes", "expected_goals", "expected_assists",
    "expected_goal_involvements", "expected_goals_conceded", "total_points", "bonus", "bps",
]
display(pd.DataFrame({
    "field": fields,
    "in_selected_merged_file": [field in merged_table for field in fields],
    "in_local_FPL_bootstrap_players": [field in bootstrap_players for field in fields],
}))

,field,in_selected_merged_file,in_local_FPL_bootstrap_players
0,goals_scored,True,True
1,assists,True,True
2,minutes,True,True
3,expected_goals,True,True
4,expected_assists,True,True
5,expected_goal_involvements,True,True
6,expected_goals_conceded,True,True
7,total_points,True,True
8,bonus,True,True
9,bps,True,True


## Saved JSON structure

Bootstrap holds player, team, gameweek and position lists plus settings; fixtures are separate match records. The archive's `latest` files are older saved snapshots. Step 8 uses a complete timestamped self-collected batch, including each player's fixture history.

In [9]:
display(json_structure)
display(table_summary.loc[table_summary["table"].str.contains(".json", regex=False)])

,filename,section,type,items
0,fpl_bootstrap_latest.json,chips,list,8
1,fpl_bootstrap_latest.json,events,list,38
2,fpl_bootstrap_latest.json,game_settings,dict,35
3,fpl_bootstrap_latest.json,game_config,dict,4
4,fpl_bootstrap_latest.json,phases,list,11
5,fpl_bootstrap_latest.json,teams,list,20
6,fpl_bootstrap_latest.json,total_players,int,<NA>
7,fpl_bootstrap_latest.json,element_stats,list,26
8,fpl_bootstrap_latest.json,element_types,list,4
9,fpl_bootstrap_latest.json,elements,list,667


,table,rows,columns
0,fpl_bootstrap_latest.json / chips,8,7
1,fpl_bootstrap_latest.json / events,38,29
2,fpl_bootstrap_latest.json / phases,11,5
3,fpl_bootstrap_latest.json / teams,20,22
4,fpl_bootstrap_latest.json / element_stats,26,2
5,fpl_bootstrap_latest.json / element_types,4,13
6,fpl_bootstrap_latest.json / elements,667,109
25,fpl_fixtures_latest.json,380,17


## Cleaning

Prepare copies of the separate player summaries, fixtures and teams. Normalize known IDs and fixture timestamps to UTC, remove exact repeated fixture/team rows and exclude `AM` summaries. Preserve zero minutes, missing values and original price units. Player summaries have no IDs, so do not deduplicate them by name.

`prepared_tables` keeps each family/season separate. `working_team_seasons` and `working_fixtures` stack their respective copies with season/source metadata and explicit FPL key names.

In [10]:
prepared_tables, cleaning_summary = prepare_fpl_tables(tables, SEASONS)
working_team_seasons = stack_seasons(prepared_tables, "teams", SEASONS).rename(columns={"id": "fpl_team_id"})
working_fixtures = stack_seasons(prepared_tables, "fixtures", SEASONS).rename(columns={
    "id": "fpl_fixture_id", "team_h": "home_fpl_team_id", "team_a": "away_fpl_team_id",
})
display(cleaning_summary)
display(prepared_tables["players", "2024_2025"].head())

,table,season,input_rows,output_rows,exact_duplicates_removed,manager_rows_removed
0,players,2022_2023,778,778,0,0
1,fixtures,2022_2023,380,380,0,0
2,teams,2022_2023,20,20,0,0
3,players,2023_2024,865,865,0,0
4,fixtures,2023_2024,380,380,0,0
5,teams,2023_2024,20,20,0,0
6,players,2024_2025,804,784,0,20
7,fixtures,2024_2025,380,380,0,0
8,teams,2024_2025,20,20,0,0
9,players,2025_2026,841,841,0,0


,first_name,second_name,goals_scored,assists,total_points,minutes,goals_conceded,creativity,influence,threat,bonus,bps,ict_index,clean_sheets,red_cards,yellow_cards,selected_by_percent,now_cost,element_type
0,Fábio,Ferreira Vieira,0,0,0,0,0,0.0000,0.0000,0.0000,0,0,0.0000,0,0,0,0.0000,54,MID
1,Gabriel,Fernando de Jesus,3,2,42,600,5,119.5000,154.4000,255.0000,6,152,52.6000,2,0,4,1.0000,65,FWD
2,Gabriel,dos Santos Magalhães,3,2,117,2363,22,208.8000,584.6000,287.0000,9,459,108.2000,10,0,4,12.2000,61,DEF
3,Kai,Havertz,9,3,97,1872,20,269.0000,467.6000,711.0000,14,343,144.8000,7,0,5,6.2000,77,FWD
4,Karl,Hein,0,0,0,0,0,0.0000,0.0000,0.0000,0,0,0.0000,0,0,0,0.0000,40,GK


### Player positions and point targets

`AM` is the 2024/25 Assistant Manager chip's manager entity, excluded from player views. Remove exact repeated match rows, exclude archived `xP` because its pre-deadline timing is unverified, and preserve other observations. See the [archive timing warning](https://github.com/vaastav/Fantasy-Premier-League#xp-column--potential-lookahead-for-ml-models).

Set **`P_off = total_points`**, including awarded bonus. Initialize `P_adj` as missing here; step 7 applies the versioned [target policy](../docs/fpl_rules.md#adjusted-target-used-in-notebook-1) after defensive validation. Historical bonus, assist definitions and FPL positions remain as recorded.

Match views use `(season, fpl_player_id, fpl_fixture_id)` and FPL club/opponent keys derived from the fixture. For a gameweek target, sum distinct fixture points. Future predictors must use information available before the target deadline.

In [11]:
player_match_views, target_view_summary = prepare_player_matches(tables, prepared_tables, SEASONS, P_ADJ_RULES)
working_player_matches = pd.concat(player_match_views.values(), ignore_index=True, sort=False)
display(target_view_summary)
display(working_player_matches[[
    "season", "name", "fpl_player_id", "fpl_fixture_id", "fpl_team_id", "opponent_fpl_team_id",
    "GW", "P_off", "P_adj", "P_adj_status",
]].head())

,season,source_rows,player_rows,exact_duplicates_removed,manager_rows_removed,calculated_P_adj_rows
0,2022_2023,26505,26505,0,0,0
1,2023_2024,29725,29725,0,0,0
2,2024_2025,27605,27283,0,322,0
3,2025_2026,29757,29747,10,0,0


,season,name,fpl_player_id,fpl_fixture_id,fpl_team_id,opponent_fpl_team_id,GW,P_off,P_adj,P_adj_status
0,2022_2023,Nathan Redmond,403,8,17,18,1,1,<NA>,not_calculated
1,2022_2023,Junior Stanislas,58,2,3,2,1,1,<NA>,not_calculated
2,2022_2023,Armando Broja,150,3,6,8,1,1,<NA>,not_calculated
3,2022_2023,Fabian Schär,366,7,15,16,1,15,<NA>,not_calculated
4,2022_2023,Jonny Evans,249,6,10,4,1,1,<NA>,not_calculated


### FPL player registry

Group name aliases and positions under each `(season, fpl_player_id)`. Lucas Digne's different seasonal IDs illustrate this scope; they do not represent different people. Cross-season continuity can use a reviewed name/alias link later without replacing source keys. No permanent internal player ID is assigned here.

In [12]:
aliases = lambda values: tuple(sorted(values.dropna().unique()))
fpl_player_registry = working_player_matches.groupby(["season", "fpl_player_id"]).agg(
    name_aliases=("name", aliases), position_labels=("position", aliases), fixture_records=("fpl_fixture_id", "size"),
).reset_index()
display(fpl_player_registry.groupby("season").size().rename("FPL_player_identities").to_frame())
display(fpl_player_registry.loc[fpl_player_registry["name_aliases"].map(lambda names: "Lucas Digne" in names)])

,FPL_player_identities
season,
2022_2023,778
2023_2024,865
2024_2025,784
2025_2026,841


,season,fpl_player_id,name_aliases,position_labels,fixture_records
32,2022_2023,33,"(Lucas Digne,)","(DEF,)",38
819,2023_2024,42,"(Lucas Digne,)","(DEF,)",38
1678,2024_2025,36,"(Lucas Digne,)","(DEF,)",38
2463,2025_2026,37,"(Lucas Digne,)","(DEF,)",38


### Step 1: FotMob identity and fixture lookups

Read names, source IDs, match-time clubs, UTC dates and status flags from the local cache. These are technical lookups for defensive enrichment; FPL remains the working identity system. No performance values or Opta IDs are extracted. Listed substitutes are not necessarily appearances; retain unusable ID-zero records separately.

In [13]:
fotmob_cache = RAW_DIR.parent / "sofascore/sofascore_matches_cache.tar.gz"
fotmob_lookups = load_fotmob_identity_lookups(fotmob_cache, SEASONS)
fotmob_fixture_lookup = fotmob_lookups["fixtures"]
fotmob_player_match_lookup = fotmob_lookups["player_matches"]
fotmob_identity_issues = fotmob_lookups["identity_issues"]
lookup_summary = pd.DataFrame([
    {"table": name, "rows": len(frame), "columns": len(frame.columns)}
    for name, frame in fotmob_lookups.items() if isinstance(frame, pd.DataFrame)
])
display(lookup_summary)
display(fotmob_identity_issues)

,table,rows,columns
0,players,1437,6
1,teams,25,6
2,fixtures,1521,11
3,player_matches,60765,7
4,team_seasons,80,3
5,identity_issues,11,9
6,source_repairs,5,9


,season,fotmob_player_id,fotmob_fixture_id,fotmob_team_id,player_name,kickoff_time,source_member,reported_fotmob_player_id,issue
0,2022_2023,<NA>,3901310,9879,Matthew Dibley-Dias,2023-05-28 15:30:00+00:00,cache_matches/3901310.json,0,missing_or_nonpositive_player_id
1,2023_2024,<NA>,4193454,9879,Matthew Dibley-Dias,2023-08-12 14:00:00+00:00,cache_matches/4193454.json,0,missing_or_nonpositive_player_id
2,2023_2024,<NA>,4193457,9937,Ehor Yarmolyuk,2023-08-13 13:00:00+00:00,cache_matches/4193457.json,0,missing_or_nonpositive_player_id
3,2023_2024,<NA>,4193472,9937,Ehor Yarmolyuk,2023-08-19 14:00:00+00:00,cache_matches/4193472.json,0,missing_or_nonpositive_player_id
4,2023_2024,<NA>,4193482,9937,Dakota Ochsenham,2023-08-26 14:00:00+00:00,cache_matches/4193482.json,0,missing_or_nonpositive_player_id
5,2023_2024,<NA>,4193482,9937,Ehor Yarmolyuk,2023-08-26 14:00:00+00:00,cache_matches/4193482.json,0,missing_or_nonpositive_player_id
6,2023_2024,<NA>,4193489,8657,Andre Brooks,2023-08-27 13:00:00+00:00,cache_matches/4193489.json,0,missing_or_nonpositive_player_id
7,2023_2024,<NA>,4193491,9937,Ehor Yarmolyuk,2023-09-02 14:00:00+00:00,cache_matches/4193491.json,0,missing_or_nonpositive_player_id
8,2023_2024,<NA>,4193505,9937,Ehor Yarmolyuk,2023-09-16 16:30:00+00:00,cache_matches/4193505.json,0,missing_or_nonpositive_player_id
9,2023_2024,<NA>,4193511,9937,Ehor Yarmolyuk,2023-09-23 16:30:00+00:00,cache_matches/4193511.json,0,missing_or_nonpositive_player_id


### Step 2: Match external teams

Match season-scoped clubs using normalized names and explicit aliases such as `Man City`/`Manchester City`. Require a one-to-one bridge. Translate club IDs only in `matching_fixtures` and `matching_players`, leaving the FPL working tables intact.

In [14]:
team_identity_map = build_team_identity_map(
    {season: prepared_tables["teams", season] for season in SEASONS}, fotmob_lookups["team_seasons"],
)
matching_fixtures, matching_players = external_match_inputs(working_player_matches, working_fixtures, team_identity_map)
display(team_identity_map.groupby("season").agg(
    source_FPL_teams=("fpl_team_id", "size"), matched_FotMob_teams=("fotmob_team_id", "nunique"),
))
display(team_identity_map.head())

,source_FPL_teams,matched_FotMob_teams
season,,
2022_2023,20,20
2023_2024,20,20
2024_2025,20,20
2025_2026,20,20


,season,fpl_team_id,fpl_team_name,fotmob_team_id,fotmob_team_name,mapping_method,mapping_status
0,2022_2023,1,Arsenal,9825,Arsenal,normalized_name,validated_team_name
1,2022_2023,2,Aston Villa,10252,Aston Villa,normalized_name,validated_team_name
2,2022_2023,3,Bournemouth,8678,AFC Bournemouth,explicit_club_alias,validated_team_name
3,2022_2023,4,Brentford,9937,Brentford,normalized_name,validated_team_name
4,2022_2023,5,Brighton,10204,Brighton & Hove Albion,explicit_club_alias,validated_team_name


### Step 3: Match external fixtures

Use season, home/away clubs and UTC kickoff. Require a unique completed, non-cancelled candidate within 30 minutes. Preserve both timestamps and send larger differences or ambiguous matches to review. Crystal Palace–West Ham on 29 April 2023 differs by 15 minutes; the abandoned Bournemouth–Luton record is excluded.

In [15]:
fixture_mapping_result = build_fixture_identity_map(matching_fixtures, fotmob_fixture_lookup, tolerance_minutes=30)
fixture_identity_map = fixture_mapping_result["mapping"]
fixture_identity_review = fixture_mapping_result["review"]
excluded_fotmob_fixtures = fotmob_fixture_lookup.loc[~fotmob_fixture_lookup["finished"] | fotmob_fixture_lookup["cancelled"]]
display(fixture_identity_map.groupby(["season", "mapping_status", "mapping_method"]).size().rename("fixtures").reset_index())
display(fixture_identity_map.loc[fixture_identity_map["kickoff_difference_minutes"].ne(0)])
display(excluded_fotmob_fixtures)
display(fixture_identity_review)

,season,mapping_status,mapping_method,fixtures
0,2022_2023,matched,same_teams_exact_kickoff,379
1,2022_2023,matched,same_teams_kickoff_tolerance,1
2,2023_2024,matched,same_teams_exact_kickoff,380
3,2024_2025,matched,same_teams_exact_kickoff,380
4,2025_2026,matched,same_teams_exact_kickoff,380


,season,fpl_fixture_id,fpl_kickoff_utc,home_fpl_team_id,away_fpl_team_id,home_fotmob_team_id,away_fotmob_team_id,fotmob_fixture_id,fotmob_kickoff_utc,kickoff_difference_minutes,mapping_status,mapping_method
324,2022_2023,335,2023-04-29 11:30:00+00:00,7,19,9826,8654,3901266,2023-04-29 11:45:00+00:00,15.0000,matched,same_teams_kickoff_tolerance


,season,fotmob_fixture_id,kickoff_time,home_fotmob_team_id,home_team_name,away_fotmob_team_id,away_team_name,finished,cancelled,status_reason,source_member
541,2023_2024,4193691,2023-12-16 15:00:00+00:00,8678,AFC Bournemouth,8346,Luton Town,True,True,Abandoned,cache_matches/4193691.json


,season,fpl_fixture_id,fpl_kickoff_utc,home_fpl_team_id,away_fpl_team_id,home_fotmob_team_id,away_fotmob_team_id,fotmob_fixture_id,fotmob_kickoff_utc,kickoff_difference_minutes,within_tolerance


### Step 4: Match external player records

Require name evidence and at least one shared fixture with the same match-time club. The helper handles normalized names, reordered/contained tokens and explicit aliases. Saved bootstrap full-name/display-name pairs supply labels only; snapshot IDs, clubs and statistics are not transferred to historical records.

Keep candidate names, methods and supporting fixtures in `player_identity_candidates`. Ambiguous or conflicting matches stay unresolved; missing match-day records never imply zero defensive actions. The bridge and review tables support enrichment without adding FotMob ID columns to `working_player_matches`.

In [16]:
fpl_snapshot_name_aliases = [
    (f"{player.first_name} {player.second_name}".strip(), alias.strip())
    for player in bootstrap_players[["first_name", "second_name", "web_name", "known_name"]].itertuples(index=False)
    for alias in (player.web_name, player.known_name) if isinstance(alias, str) and alias.strip()
]
player_mapping_result = build_player_identity_map(
    matching_players, fixture_identity_map, fotmob_player_match_lookup,
    fotmob_identity_issues=fotmob_identity_issues,
    reviewed_name_aliases=PLAYER_NAME_ALIASES, source_name_aliases=fpl_snapshot_name_aliases,
)
player_identity_map = player_mapping_result["mapping"]
player_identity_candidates = player_mapping_result["candidates"]
player_source_id_issues = player_mapping_result["source_id_issues"]
player_report = player_matching_report(working_player_matches, player_mapping_result)
player_identity_review, player_mapping_summary = player_report["review"], player_report["summary"]
identity_columns = ["season", "fpl_player_id", "name_aliases", "fotmob_player_id", "mapping_status", "shared_fixtures", "mapping_method"]
display(player_identity_map.groupby(["season", "mapping_status"]).size().rename("identities").reset_index())
display(player_mapping_summary)
display(player_identity_map[identity_columns].head())
display(player_identity_review.loc[player_identity_review["recorded_minutes"].gt(0)])
display(player_identity_review[identity_columns + ["recorded_minutes"]].head())
display(player_report["conflicts"])
display(player_source_id_issues)

,season,mapping_status,identities
0,2022_2023,matched,691
1,2022_2023,no_shared_fixture_name_match,86
2,2022_2023,unusable_fotmob_source_id,1
3,2023_2024,matched,723
4,2023_2024,no_shared_fixture_name_match,141
5,2023_2024,unusable_fotmob_source_id,1
6,2024_2025,matched,678
7,2024_2025,no_shared_fixture_name_match,106
8,2025_2026,matched,674
9,2025_2026,no_shared_fixture_name_match,167


,player_fixture_rows,proposed_matched_rows,positive_minute_rows,matched_positive_minute_rows
season,,,,
2022_2023,26505,23358,11345,11345
2023_2024,29725,24444,11384,11384
2024_2025,27283,23325,11567,11567
2025_2026,29747,23442,11492,11492


,season,fpl_player_id,name_aliases,fotmob_player_id,mapping_status,shared_fixtures,mapping_method
0,2022_2023,1,"(Cédric Alves Soares,)",160627,matched,27,contained_name_tokens
1,2022_2023,2,"(Bernd Leno,)",215168,matched,38,normalized_name
2,2022_2023,3,"(Granit Xhaka,)",207236,matched,37,normalized_name
3,2022_2023,4,"(Mohamed Elneny,)",339992,matched,9,normalized_name
4,2022_2023,5,"(Rob Holding,)",626667,matched,38,normalized_name


,season,fpl_player_id,name_aliases,fotmob_player_id,candidate_fotmob_ids,candidate_count,shared_fixtures,evidence_fixture_ids,mapping_method,mapping_status,recorded_minutes


,season,fpl_player_id,name_aliases,fotmob_player_id,mapping_status,shared_fixtures,mapping_method,recorded_minutes
0,2022_2023,17,"(Nuno Varela Tavares,)",<NA>,no_shared_fixture_name_match,0,none,0
1,2022_2023,20,"(Pablo Marí Villar,)",<NA>,no_shared_fixture_name_match,0,none,0
2,2022_2023,22,"(Lucas Torreira di Pascua,)",<NA>,no_shared_fixture_name_match,0,none,0
3,2022_2023,38,"(Kortney Hause,)",<NA>,no_shared_fixture_name_match,0,none,0
4,2022_2023,51,"(Anwar El Ghazi,)",<NA>,no_shared_fixture_name_match,0,none,0


,season,fpl_player_id,fotmob_player_id,fotmob_team_id,shared_fixture_ids,shared_fixtures,name_evidence


,season,fpl_player_id,fpl_name,source_name,fotmob_fixture_id,issue
0,2022_2023,763,Matthew Dibley-Dias,Matthew Dibley-Dias,3901310,missing_or_nonpositive_player_id
1,2023_2024,120,Yegor Yarmolyuk,Ehor Yarmolyuk,4193457,missing_or_nonpositive_player_id
2,2023_2024,652,Matthew Dibley-Dias,Matthew Dibley-Dias,4193454,missing_or_nonpositive_player_id
3,2023_2024,120,Yegor Yarmolyuk,Ehor Yarmolyuk,4193472,missing_or_nonpositive_player_id
4,2023_2024,655,Andre Brooks,Andre Brooks,4193489,missing_or_nonpositive_player_id
5,2023_2024,120,Yegor Yarmolyuk,Ehor Yarmolyuk,4193482,missing_or_nonpositive_player_id
6,2023_2024,120,Yegor Yarmolyuk,Ehor Yarmolyuk,4193491,missing_or_nonpositive_player_id
7,2023_2024,120,Yegor Yarmolyuk,Ehor Yarmolyuk,4193505,missing_or_nonpositive_player_id
8,2023_2024,655,Andre Brooks,Andre Brooks,4193519,missing_or_nonpositive_player_id
9,2023_2024,120,Yegor Yarmolyuk,Ehor Yarmolyuk,4193511,missing_or_nonpositive_player_id


### Missing defensive fields across seasons

The four defensive fields below are absent in 2022/23–2024/25 and supplied in 2025/26. Preserve unavailable measurements as missing. In the latter file, `defensive_contribution` is an action count: CBI plus tackles for defenders, plus recoveries for midfielders/forwards; goalkeepers have zero.

The [2025/26 defensive reward](https://www.premierleague.com/en/news/4362780/fantasy-premier-league-202526-is-live) began that season, so `P_off` includes it only where applicable. FotMob reconstruction needs compatible definitions and validation before older records can receive an adjusted label.

In [17]:
defensive_fields = ["clearances_blocks_interceptions", "defensive_contribution", "recoveries", "tackles"]
defensive_field_coverage = pd.DataFrame({
    season: {
        field: f"present; {tables[f'fpl_merged_gw_{season}.csv'][field].isna().sum()} nulls"
        if field in tables[f"fpl_merged_gw_{season}.csv"] else "not supplied"
        for field in defensive_fields
    } for season in SEASONS
})
display(defensive_field_coverage)

latest_matches = player_match_views["2025_2026"]
count = latest_matches["clearances_blocks_interceptions"] + latest_matches["tackles"]
count += latest_matches["recoveries"].where(latest_matches["position"].isin(["MID", "FWD"]), 0)
count = count.where(latest_matches["position"].ne("GK"), 0)
count_checks = latest_matches.assign(count_formula_mismatches=latest_matches["defensive_contribution"].ne(count)).groupby("position").agg(
    rows=("position", "size"), count_formula_mismatches=("count_formula_mismatches", "sum"),
).reset_index()
display(count_checks)

,2022_2023,2023_2024,2024_2025,2025_2026
clearances_blocks_interceptions,not supplied,not supplied,not supplied,present; 0 nulls
defensive_contribution,not supplied,not supplied,not supplied,present; 0 nulls
recoveries,not supplied,not supplied,not supplied,present; 0 nulls
tackles,not supplied,not supplied,not supplied,present; 0 nulls


,position,rows,count_formula_mismatches
0,DEF,9733,0
1,FWD,3278,0
2,GK,3427,0
3,MID,13309,0


### Step 5: Extract defensive actions

Read clearances, defender `shot_blocks`, interceptions, tackles and recoveries from completed FotMob fixtures. Deduplicate repeated metric displays and preserve missing measurements. Check reviewed identity repairs against the same FotMob lineups/events and log original cells separately. Keep other invalid IDs and conflicting values for review.

Build `defensive_diagnostics` through the technical player, fixture and match-time club bridges. This comparison table holds external values separately from the FPL working data. `complete` describes field coverage, not exact agreement with FPL.

In [18]:
from src.defensive_contributions import (
    ENRICHMENT_COLUMNS, FPL_DEFENSIVE_FIELDS, STAT_COLUMNS, attach_defensive_actions,
    fill_historical_defensive_fields, load_defensive_actions, validate_defensive_actions,
)

defensive_extraction = load_defensive_actions(fotmob_cache, fotmob_fixture_lookup)
defensive_actions = defensive_extraction["actions"]
defensive_attachment = attach_defensive_actions(
    working_player_matches, defensive_actions, fixture_identity_map, player_identity_map, team_identity_map,
)
defensive_diagnostics = defensive_attachment["matches"]
defensive_coverage = defensive_attachment["coverage"]
defensive_attachment_review = defensive_attachment["review"]
display(defensive_coverage)
display(defensive_diagnostics[MATCH_KEY + ["name"] + STAT_COLUMNS + ["defensive_status"]].head())
print("Unresolved defensive appearances:", len(defensive_attachment_review))
if not defensive_attachment_review.empty:
    display(defensive_attachment_review[MATCH_KEY + ["name", "minutes", "external_minutes", "defensive_status"]])
source_id_review = defensive_extraction["source_issues"]
print("Unusable source IDs with positive minutes:", int(source_id_review["external_minutes"].gt(0).sum()))
fotmob_source_repairs = defensive_extraction["source_repairs"]
display(fotmob_source_repairs[[
    "fotmob_fixture_id", "reported_player_id", "reported_name",
    "fotmob_player_id", "player_name", "substitution_minutes",
]])
print("Conflicting or invalid defensive metrics:", len(defensive_extraction["stat_issues"]))

,player_fixture_rows,source_records,positive_minute_rows,positive_outfield_rows,complete_outfield_rows
season,,,,,
2022_2023,26505,15145,11345,10575,10575
2023_2024,29725,15143,11384,10608,10608
2024_2025,27283,15166,11567,10797,10797
2025_2026,29747,15181,11492,10725,10725


,season,fpl_player_id,fpl_fixture_id,name,external_minutes,external_clearances,external_shot_blocks,external_interceptions,external_tackles,external_recoveries,defensive_status
0,2022_2023,403,8,Nathan Redmond,1,0,0,0,0,0,complete
1,2022_2023,58,2,Junior Stanislas,1,0,0,0,0,0,complete
2,2022_2023,150,3,Armando Broja,15,0,0,0,0,2,complete
3,2022_2023,366,7,Fabian Schär,90,4,2,3,2,8,complete
4,2022_2023,249,6,Jonny Evans,90,5,0,0,0,5,complete


Unresolved defensive appearances: 0
Unusable source IDs with positive minutes: 0


,fotmob_fixture_id,reported_player_id,reported_name,fotmob_player_id,player_name,substitution_minutes
0,4193455,0,Andre Brooks,1396784,Andre Brooks,"(87,)"
1,4193476,0,Andre Brooks,1396784,Andre Brooks,"(74,)"
2,4193491,0,Dakota Ochsenham,1130797,Michael Olakigbe,"(82,)"
3,4193500,1553377,Alex Matos,1202031,Alex Matos,()
4,4193511,0,Dakota Ochsenham,1130797,Michael Olakigbe,"(89,)"


Conflicting or invalid defensive metrics: 0


### Step 6: Validate defensive reconstruction

Compare 2025/26 outfield appearances with FPL counts and cross-check supplied fixture counts. Apply the [frozen 10/12-action thresholds](../docs/fpl_rules.md#defensive-contribution) using each row's FPL position, capped at two points per fixture. `DC_reference` is derived from FPL counts, not independently verified live award breakdowns.

Report agreement and every award disagreement in separate review tables. These comparisons assess historical estimates; official FPL values remain authoritative throughout 2025/26.

In [19]:
defensive_validation = validate_defensive_actions(defensive_diagnostics, working_fixtures)
defensive_action_review = defensive_validation["disagreements"]
defensive_award_review = defensive_validation["award_disagreements"]
display(defensive_validation["summary"])
display(defensive_validation["awards"])
display(defensive_validation["fixture_reference_summary"])
display(defensive_award_review[[
    "name", "fpl_fixture_id", "position", "defensive_contribution", "external_defensive_count",
    "DC_reference", "DC_estimated", "near_threshold",
]])
display(defensive_award_review[[
    "external_cbi_difference", "external_tackles_difference", "external_recoveries_difference",
]].ne(0).sum().rename("award_disagreement_cases").to_frame())
print("Action-disagreement records:", len(defensive_action_review))
print("Unavailable overlap records:", len(defensive_validation["unavailable"]))

,metric,compared,equal,mismatches,agreement,absolute_error_sum
0,external_cbi,10725,10511,214,0.9800,239
1,external_tackles,10725,10632,93,0.9913,94
2,external_recoveries,10725,10608,117,0.9891,121
3,external_defensive_count,10725,10388,337,0.9686,378
4,DC_estimated,10725,10698,27,0.9975,54


,position,compared,reference_awards,estimated_awards,correct_positive,false_positive,false_negative,precision,recall
0,ALL,10725,1417,1434,1412,22,5,0.9847,0.9965
1,DEF,3950,821,833,818,15,3,0.9820,0.9963
2,FWD,1428,9,9,9,0,0,1.0000,1.0000
3,MID,5347,587,592,585,7,2,0.9882,0.9966


,compared,mismatches,positive_counts_without_fixture_entry
0,9721,0,0


,name,fpl_fixture_id,position,defensive_contribution,external_defensive_count,DC_reference,DC_estimated,near_threshold
84827,Bruno Guimarães Rodriguez Moura,19,MID,11.0000,12,0,2,True
85202,Maxence Lacroix,21,DEF,9.0000,10,0,2,True
85898,Matthijs de Ligt,38,DEF,9.0000,11,0,2,True
86550,Tijjani Reijnders,41,MID,12.0000,11,2,0,True
86558,Tyler Adams,42,MID,11.0000,12,0,2,True
89663,Dan Ndoye,83,MID,11.0000,12,0,2,True
89740,Chris Richards,81,DEF,9.0000,10,0,2,True
89798,Jayden Bogle,87,DEF,9.0000,10,0,2,True
89856,Alex Scott,83,MID,11.0000,12,0,2,True
90373,Daniel Ballard,98,DEF,10.0000,9,2,0,True


,award_disagreement_cases
external_cbi_difference,19
external_tackles_difference,6
external_recoveries_difference,3


Action-disagreement records: 382
Unavailable overlap records: 0


### Fill historical FPL defensive fields

Fill only missing `clearances_blocks_interceptions`, `defensive_contribution`, `recoveries` and `tackles` in the 2022/23–2024/25 working records. Derive the contribution count from the prepared components and each row's FPL position. Keep every 2025/26 FPL value unchanged, including provider disagreements.

The working schema has no external diagnostic columns. `defensive_fill_audit` records filled fields, source members and methods separately. Goalkeeper contribution counts are zero by rule; unknown action measurements and unresolved appearances remain missing.

In [20]:
official_2025 = working_player_matches.loc[working_player_matches["season"].eq("2025_2026")].copy()
defensive_fill = fill_historical_defensive_fields(working_player_matches, defensive_diagnostics)
working_player_matches = defensive_fill["matches"]
defensive_fill_audit, defensive_fill_summary = defensive_fill["audit"], defensive_fill["summary"]
pd.testing.assert_frame_equal(
    official_2025, working_player_matches.loc[working_player_matches["season"].eq("2025_2026")],
)
display(defensive_fill_summary)
display(working_player_matches[MATCH_KEY + ["name"] + list(FPL_DEFENSIVE_FIELDS) + ["P_off", "P_adj"]].head())

,clearances_blocks_interceptions,defensive_contribution,recoveries,tackles
season,,,,
2022_2023,10575,13366,11345,11345
2023_2024,10608,14021,11384,11384
2024_2025,10797,13666,11567,11567
2025_2026,0,0,0,0


,season,fpl_player_id,fpl_fixture_id,name,clearances_blocks_interceptions,defensive_contribution,recoveries,tackles,P_off,P_adj
0,2022_2023,403,8,Nathan Redmond,0.0000,0.0000,0.0000,0.0000,1,<NA>
1,2022_2023,58,2,Junior Stanislas,0.0000,0.0000,0.0000,0.0000,1,<NA>
2,2022_2023,150,3,Armando Broja,0.0000,2.0000,2.0000,0.0000,1,<NA>
3,2022_2023,366,7,Fabian Schär,9.0000,11.0000,8.0000,2.0000,15,<NA>
4,2022_2023,249,6,Jonny Evans,5.0000,5.0000,5.0000,0.0000,1,<NA>


### Step 7: Calculate adjusted points

Apply the [versioned rules](../docs/fpl_rules.md#adjusted-target-used-in-notebook-1): add historical defensive awards, preserve official defensive scoring from 2025/26, and add Ferguson's verified appearance point once. `P_off` remains unchanged. Missing counts for historical appearances leave an incomplete target; zero-minute records do not require invented action measurements.

Keep the calculation components in `point_adjustment_audit`. Only the reserved `P_adj`, status and rule-version fields change in the working table.

In [21]:
from src.defensive_contributions import calculate_adjusted_points

point_adjustment = calculate_adjusted_points(working_player_matches)
working_player_matches = point_adjustment["matches"]
point_adjustment_audit = point_adjustment["audit"]
display(point_adjustment["summary"])
display(point_adjustment_audit.loc[
    point_adjustment_audit["verified_point_correction"].ne(0),
    MATCH_KEY + ["P_off", "DC_adjustment", "verified_point_correction", "P_adj"],
])

,rows,adjusted_rows,DC_points_added,verified_point_corrections,P_off_total,P_adj_total
season,,,,,,
2022_2023,26505,26505,2634,0,31724,"34,358.0000"
2023_2024,29725,29725,2530,0,31271,"33,801.0000"
2024_2025,27283,27283,2506,1,31290,"33,797.0000"
2025_2026,29747,29747,0,0,34382,"34,382.0000"


,season,fpl_player_id,fpl_fixture_id,P_off,DC_adjustment,verified_point_correction,P_adj
71993,2024_2025,123,239,0,0,1,1.0000


### Step 8: Join the collected current season

Verify the complete batch's response checksums and reconcile its player histories. Join names and positions by `(season, fpl_player_id)`, then club names by `(season, fpl_team_id)` using each match's club. Keep native IDs and collection timestamps; cumulative snapshot statistics stay separate.

`current_player_matches` retains official `P_off` without recalculation. Historical `P_off` and `P_adj` keep their existing meanings. No current-season defensive award is added again.

In [22]:
CURRENT_SEASON = "2026_2027"
COLLECTION_BATCH = PROJECT_ROOT / "data/raw/fpl_api" / CURRENT_SEASON / "20261110T210622560387Z"
collected = prepare_collected_batch(COLLECTION_BATCH, CURRENT_SEASON)
current_tables = collected["tables"]
current_player_snapshot = current_tables["current_players"]
current_player_matches = current_tables["current_player_matches"]
current_team_snapshot = current_tables["current_teams"]
current_fixtures = current_tables["current_fixtures"]
current_provisional_fixtures = current_fixtures.loc[current_fixtures.started.fillna(False) & ~current_fixtures.finished]
display(current_tables["current_snapshot_metadata"])
display(collected["reconciliation"].groupby("field").agg(
    players=("fpl_player_id", "size"), disagreements=("matches", lambda values: (~values).sum()),
))
display(current_player_matches[MATCH_KEY + [
    "name", "position", "team", "minutes", "P_off", "expected_goals", "expected_assists",
]].head())

,season,retrieved_at_utc,price_updated_at_utc,last_finished_kickoff_utc,last_started_kickoff_utc,last_checked_GW,players,finished_fixtures,started_unfinished_fixtures,player_history_available,histories_collected,player_fixture_rows,collection_batch
0,2026_2027,2026-11-10 21:06:23.051534+00:00,2026-11-10 21:05:12.660965+00:00,2026-11-08 16:30:00+00:00,2026-11-08 16:30:00+00:00,10,672,100,0,True,672,6562,20261110T210622560387Z


,players,disagreements
field,,
assists,672,0
clearances_blocks_interceptions,672,0
defensive_contribution,672,0
expected_assists,672,0
expected_goals,672,0
goals_scored,672,0
minutes,672,0
recoveries,672,0
starts,672,0


,season,fpl_player_id,fpl_fixture_id,name,position,team,minutes,P_off,expected_goals,expected_assists
0,2026_2027,1,1,David Raya Martín,GK,Arsenal,90,6,0.0000,0.0000
1,2026_2027,1,20,David Raya Martín,GK,Arsenal,90,6,0.0000,0.0000
2,2026_2027,1,29,David Raya Martín,GK,Arsenal,90,3,0.0000,0.0100
3,2026_2027,1,36,David Raya Martín,GK,Arsenal,90,14,0.0000,0.0000
4,2026_2027,1,42,David Raya Martín,GK,Arsenal,90,1,0.0000,0.0000


### Step 9: Finalize coverage and totals

Check native keys, match-time clubs, fixture context and point labels. Report nulls, duplicates and source-total differences separately. Historical and current match tables share fixture-level observations; cumulative snapshots remain separate. Current points use `P_off`; `P_adj` is reserved for the historical adjustment.

In [23]:
working_player_matches[list(FPL_DEFENSIVE_FIELDS)] = working_player_matches[list(FPL_DEFENSIVE_FIELDS)].astype("Int64")
fixture_context_review = validate_fpl_relations(working_player_matches, working_fixtures, working_team_seasons)
assert len(fpl_player_registry) == working_player_matches.groupby(["season", "fpl_player_id"]).ngroups
assert not set(ENRICHMENT_COLUMNS).intersection(working_player_matches.columns)
assert len(working_player_matches) == target_view_summary.player_rows.sum()
assert working_player_matches.P_adj.notna().all()
assert working_player_matches.P_adj_rules.eq(ADJUSTED_RULES).all()
assert not working_player_matches.P_adj_status.isin(["incomplete", "not_calculated"]).any()
assert working_player_matches.loc[working_player_matches.season.eq("2025_2026"), "P_adj"].eq(official_2025.P_off).all()
assert not fixture_identity_map.fotmob_fixture_id.dropna().isin(excluded_fotmob_fixtures.fotmob_fixture_id).any()
assert "P_adj" not in current_player_matches

player_match_views = {season: frame.reset_index(drop=True) for season, frame in working_player_matches.groupby("season")}
coverage_columns = ["season", "fpl_player_id", "fpl_fixture_id", "minutes", "kickoff_time", "P_off", "P_adj"]
coverage_input = pd.concat([
    frame.reindex(columns=coverage_columns) for frame in [working_player_matches, current_player_matches]
], ignore_index=True)
season_coverage = coverage_input.groupby("season").agg(
    player_fixture_rows=("fpl_fixture_id", "size"), players=("fpl_player_id", "nunique"),
    fixtures=("fpl_fixture_id", "nunique"), appearances=("minutes", lambda values: values.gt(0).sum()),
    first_kickoff=("kickoff_time", "min"), last_kickoff=("kickoff_time", "max"),
    P_off=("P_off", "sum"), P_adj=("P_adj", lambda values: values.sum(min_count=1)),
)
display(season_coverage)
print("Fixture-context differences:", len(fixture_context_review) + len(collected["context_review"]))

,player_fixture_rows,players,fixtures,appearances,first_kickoff,last_kickoff,P_off,P_adj
season,,,,,,,,
2022_2023,26505,778,380,11345,2022-08-05 19:00:00+00:00,2023-05-28 15:30:00+00:00,31724,"34,358.0000"
2023_2024,29725,865,380,11384,2023-08-11 19:00:00+00:00,2024-05-19 15:00:00+00:00,31271,"33,801.0000"
2024_2025,27283,784,380,11567,2024-08-16 19:00:00+00:00,2025-05-25 15:00:00+00:00,31290,"33,797.0000"
2025_2026,29747,841,380,11492,2025-08-15 19:00:00+00:00,2026-05-24 15:00:00+00:00,34382,"34,382.0000"
2026_2027,6562,672,100,3084,2026-08-21 19:00:00+00:00,2026-11-08 16:30:00+00:00,9145,<NA>


Fixture-context differences: 0


### Reconcile source totals

Compare historical totals without joining ID-less summaries by name. The 2024/25 summary exceeds match-file official points by one; Ferguson's reviewed correction enters historical `P_adj` only.

Collected current histories reconcile with their captured bootstrap. Player goals plus own goals reconcile to fixture goals in all five seasons; preserve finality flags for subsequent updates.

In [24]:
working_player_summaries = stack_seasons(prepared_tables, "players", SEASONS)
total_reconciliation = reconcile_player_totals(working_player_matches, working_player_summaries)
goal_reconciliation = []
for season, players, fixtures in [
    *[(season, player_match_views[season], working_fixtures.loc[working_fixtures.season.eq(season)]) for season in SEASONS],
    (CURRENT_SEASON, current_player_snapshot, current_fixtures),
]:
    finalized = fixtures.loc[fixtures.finished, ["team_h_score", "team_a_score"]].sum().sum()
    recorded = fixtures[["team_h_score", "team_a_score"]].sum().sum()
    goal_reconciliation.append({
        "season": season, "player_goals_plus_own_goals": players.goals_scored.sum() + players.own_goals.sum(),
        "finalized_fixture_goals": finalized, "provisional_fixture_goals": recorded - finalized,
        "difference": players.goals_scored.sum() + players.own_goals.sum() - recorded,
    })
goal_reconciliation = pd.DataFrame(goal_reconciliation)
assert goal_reconciliation.difference.eq(0).all()
display(total_reconciliation.loc[total_reconciliation.difference.ne(0)])
display(goal_reconciliation)

,season,metric,season_summary,fixture_records,difference
17,2024_2025,total_points,31291,31290,-1


,season,player_goals_plus_own_goals,finalized_fixture_goals,provisional_fixture_goals,difference
0,2022_2023,1084,"1,084.0000",0.0000,0.0000
1,2023_2024,1246,"1,246.0000",0.0000,0.0000
2,2024_2025,1115,"1,115.0000",0.0000,0.0000
3,2025_2026,1045,"1,045.0000",0.0000,0.0000
4,2026_2027,273,273.0000,0.0000,0.0000


### Save the EDA inputs

Save typed tables, separate audits and a manifest with source checksums under `data/processed/`. Re-running replaces these prepared outputs. Raw files remain unchanged; snapshot totals and historical fixture records remain separate.

In [25]:
eda_tables = {
    "player_matches": working_player_matches, "player_registry": fpl_player_registry,
    "player_summaries": working_player_summaries, "teams": working_team_seasons,
    "fixtures": working_fixtures, **current_tables,
}
final_quality, final_column_profile = profile_tables(eda_tables)
final_keys = {
    "player_matches": MATCH_KEY, "player_registry": ["season", "fpl_player_id"],
    "teams": ["season", "fpl_team_id"], "fixtures": ["season", "fpl_fixture_id"],
    "current_player_matches": MATCH_KEY,
    "current_players": ["season", "fpl_player_id"], "current_teams": ["season", "fpl_team_id"],
    "current_fixtures": ["season", "fpl_fixture_id"], "current_events": ["season", "id"],
    "current_snapshot_metadata": ["season"],
}
for name, key in final_keys.items():
    frame = eda_tables[name]
    require_keys(frame, key, name)
    final_quality.loc[final_quality.table.eq(name), ["candidate_key", "null_key_rows", "extra_key_duplicates"]] = [", ".join(key), 0, 0]
assert not any("fotmob" in column for frame in eda_tables.values() for column in frame)
display(final_quality[["table", "rows", "columns", "null_cells", "extra_exact_duplicates"]])
display(final_column_profile.loc[final_column_profile.nulls.gt(0), ["table", "column", "dtype", "nulls", "null_share"]])

preparation_audits = {
    "season_coverage": season_coverage, "table_quality": final_quality, "column_profile": final_column_profile,
    "total_reconciliation": total_reconciliation, "goal_reconciliation": goal_reconciliation,
    "fixture_context_review": fixture_context_review, "point_adjustment": point_adjustment_audit,
    "defensive_fill": defensive_fill_audit, "defensive_coverage": defensive_coverage,
    "defensive_count_disagreements": defensive_action_review, "defensive_award_disagreements": defensive_award_review,
    "player_identity_review": player_identity_review, "source_repairs": fotmob_source_repairs,
    "current_provisional_fixtures": current_provisional_fixtures,
    "current_reconciliation": collected["reconciliation"], "current_fixture_context": collected["context_review"],
    "current_collection_requests": collected["requests"],
}
PROCESSED_DIR = PROJECT_ROOT / "data/processed"
source_files = [
    *[path for path in RAW_DIR.iterdir() if path.suffix in {".csv", ".json"}], fotmob_cache,
    *COLLECTION_BATCH.rglob("*.json"),
    *[PROJECT_ROOT / "src" / name for name in [
        "data_preparation.py", "fotmob_identity.py", "defensive_contributions.py", "collect_fpl.py",
    ]],
    PROJECT_ROOT / "docs/fpl_rules.md",
]
display(export_prepared_data(PROCESSED_DIR, eda_tables, preparation_audits, source_files, PROJECT_ROOT, ADJUSTED_RULES))

,table,rows,columns,null_cells,extra_exact_duplicates
0,player_matches,113260,58,949742,0
1,player_registry,3268,5,0,0
2,player_summaries,3268,22,2427,0
3,teams,80,24,240,0
4,fixtures,1520,19,0,0
5,current_players,672,112,4055,0
6,current_teams,20,25,60,0
7,current_fixtures,380,20,560,0
8,current_events,38,32,262,0
9,current_snapshot_metadata,1,13,0,0


,table,column,dtype,nulls,null_share
46,player_matches,mng_clean_sheets,float64,100155,0.8843
47,player_matches,mng_draw,float64,100155,0.8843
48,player_matches,mng_goals_scored,float64,100155,0.8843
49,player_matches,mng_loss,float64,100155,0.8843
50,player_matches,mng_underdog_draw,float64,100155,0.8843
51,player_matches,mng_underdog_win,float64,100155,0.8843
52,player_matches,mng_win,float64,100155,0.8843
53,player_matches,modified,object,56230,0.4965
54,player_matches,clearances_blocks_interceptions,Int64,51533,0.4550
55,player_matches,defensive_contribution,Int64,42460,0.3749


,rows,columns
table,,
player_matches,113260,58
player_registry,3268,5
player_summaries,3268,22
teams,80,24
fixtures,1520,19
current_players,672,112
current_teams,20,25
current_fixtures,380,20
current_events,38,32


## Ready for EDA

Load `data/processed/fpl_tables.pkl` with `pd.read_pickle` in Notebook 2. `player_matches` contains historical records; `current_player_matches` contains collected 2026/27 histories with official points. Player snapshots, teams and fixtures remain separate. Quality checks are in `fpl_audits.pkl`; `fpl_manifest.json` records provenance.

Start with season-by-season player and team exploration, keeping the partial current season labelled by coverage. Cross-season identity links and model features remain later work. Consult the [rules reference](../docs/fpl_rules.md) before changing scoring.